In [ ]:
#https://www.kaggle.com/datasets/wyattowalsh/basketball/data
#https://github.com/mpope9/nba-sql/blob/master/image/NBA-ER.jpg

In [1]:
from urllib.request import urlopen
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
import warnings
import re
import psycopg2

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
options = webdriver.FirefoxOptions()
options.add_argument('-headless')
driver = webdriver.Firefox(options = options)

The geckodriver version (0.33.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (128.0.3.199); currently, geckodriver 0.34.0 is recommended for firefox 128.*, so it is advised to delete the driver in PATH and retry


In [ ]:
try:
    # Establece la conexión con la base de datos
    connection = psycopg2.connect(
        dbname="mydatabase",
        user="myuser",
        password="mysecretpassword",
        host="localhost",  # normalmente es 'localhost' si es local
        port="5432"  # normalmente es 5432
    )
    
    # Crea un cursor para realizar operaciones en la base de datos
    cursor = connection.cursor()
except Exception as error:
    print(f"Error al conectar con la base de datos: {error}")

In [5]:
def getTableIDS(url):
    driver.get(url)
    tables_id = driver.find_elements(By.XPATH, "//table[@id]")
    list_id_tables = []
    for table in tables_id:
        table_id = table.get_attribute("id")
        list_id_tables.append(table_id)
    return list_id_tables

In [4]:
def delete_unnamed_columns(df):
    df = df.loc[:, ~df.columns.str.contains('Unnamed')]
    return df

In [5]:
def get_keys_dictionary(diccionario):
    keys = set(diccionario.keys())
    for values in diccionario.values():
        if isinstance(values, dict):
            keys.update(get_keys_dictionary(values))
    return keys

In [6]:
def check_missing_values(dictionary):
    for key, value in dictionary.items():
        if isinstance(value, dict):
            print(f"Recorriendo diccionario bajo la clave '{key}':")
            check_missing_values(value)  
        elif isinstance(value, pd.DataFrame): 
            print(f"Revisando DataFrame bajo la clave '{key}':")
            
            if value.isnull().values.any():
                print("¡Hay valores nulos en el DataFrame!")
                print(value)
            
            unnamed_columns = [col for col in value.columns if 'Unnamed' in col]
            if unnamed_columns:
                print(f"¡El DataFrame tiene columnas 'Unnamed': {unnamed_columns}")

            empty_columns = [col for col in value.columns if value[col].empty]
            if empty_columns:
                print(f"¡El DataFrame tiene columnas vacías: {empty_columns}")

In [ ]:
# NBA Standings que es como quedó la season con todos los equipos
years = list(range(2022, 2023))
dictionary_of_teams = {}
# Itera a través de cada identificador de tabla y guarda en un DataFrame
dataframes = []
for year in years:
    dictionary_of_teams[year] = {}
    url = f'https://www.basketball-reference.com/leagues/NBA_{year}_standings.html'
    table_ids = getTableIDS(url)
    time.sleep(3)
    for table_id in table_ids:
        dictionary_of_teams[year][table_id] = {}
        table_element = driver.find_element(By.ID, table_id)
        table_html = table_element.get_attribute('outerHTML')
        df = pd.read_html(table_html, header=0)[0]
        if table_id == 'expanded_standings':
            new_header = df.iloc[0]
            df = df[1:]
            df.columns = new_header
            df.reset_index(drop=True, inplace=True)
            dictionary_of_teams[year][table_id] = df
        else:    
            dictionary_of_teams[year][table_id] = df

In [ ]:
#De aqui para arriba tenemos las estadisticas generales de los equipos
#De aqui para abajo sacaremos las estadisticas de los jugadores por equipo

In [11]:
years = list(range(2022, 2023))
dictionary_of_players = {}
dictionary_of_players_playoffs = {}
teams_NBA_list = ['ATL']
# teams_NBA_list =  ['ATL', 'BOS', 'BRK', 'CHO', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND','LAC','LAL','MEM','MIA','MIL','MIN','NOP', 'NYK','OKC', 'ORL','PHI', 
#                    'PHO', 'POR','SAC','SAS','TOR','UTA','WAS']

for team in teams_NBA_list:
    dictionary_of_players[team] = {}
    dictionary_of_players_playoffs[team] = {}
    for year in years:
        dictionary_of_players[team][year] = {}
        dictionary_of_players_playoffs[team][year] = {}
        url = f'https://www.basketball-reference.com/teams/{team}/{year}.html'
        table_ids = getTableIDS(url)
        time.sleep(3)
        for table_id in table_ids:
            if 'playoffs' in table_id:
                dictionary_of_players_playoffs[team][year][table_id] = {}
                table_element = driver.find_element(By.ID, table_id)
                table_html = table_element.get_attribute('outerHTML')
                df = pd.read_html(table_html, header=0)[0]    
                if  table_id == 'playoffs_pbp':
                    new_header = df.iloc[0]
                    df = df[1:]
                    df.columns = new_header
                    df.reset_index(drop=True, inplace=True)
                else:    
                    dictionary_of_players_playoffs[team][year][table_id] = df
            else:
                dictionary_of_players[team][year][table_id] = {}
                table_element = driver.find_element(By.ID, table_id)
                table_html = table_element.get_attribute('outerHTML')
                df = pd.read_html(table_html, header=0)[0]    
                if table_id == 'adj_shooting' or table_id == 'shooting' or table_id == 'pbp':
                    new_header = df.iloc[0]
                    df = df[1:]
                    df.columns = new_header
                    df.reset_index(drop=True, inplace=True)
                else:    
                    dictionary_of_players[team][year][table_id] = df
        

In [12]:
dictionary_of_players['ATL'][2022]['per_game']

,Rk,Player,Age,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1,Trae Young,23,76,76,34.9,9.4,20.3,0.460,3.1,8.0,0.382,6.3,12.3,0.512,0.536,6.6,7.3,0.904,0.7,3.1,3.7,9.7,0.9,0.1,4.0,1.7,28.4
1,2,John Collins,24,54,53,30.8,6.3,11.9,0.526,1.2,3.3,0.364,5.1,8.7,0.588,0.576,2.5,3.1,0.793,1.7,6.1,7.8,1.8,0.6,1.0,1.1,3.0,16.2
2,3,De'Andre Hunter,24,53,52,29.8,4.8,10.8,0.442,1.4,3.7,0.379,3.4,7.2,0.474,0.506,2.4,3.1,0.765,0.5,2.8,3.3,1.3,0.7,0.4,1.3,2.9,13.4
3,4,Kevin Huerter,23,74,60,29.6,4.7,10.3,0.454,2.2,5.6,0.389,2.5,4.7,0.530,0.559,0.6,0.7,0.808,0.4,3.0,3.4,2.7,0.7,0.4,1.2,2.5,12.1
4,5,Bogdan Bogdanović,29,63,27,29.3,5.4,12.6,0.431,2.7,7.3,0.368,2.7,5.3,0.516,0.537,1.5,1.8,0.843,0.5,3.5,4.0,3.1,1.1,0.2,1.1,2.1,15.1
5,6,Chaundee Brown Jr.,23,3,2,27.7,3.0,8.3,0.360,2.0,5.0,0.400,1.0,3.3,0.300,0.480,1.7,2.0,0.833,0.3,4.3,4.7,1.3,0.7,0.0,0.0,2.3,9.7
6,7,Clint Capela,27,74,73,27.6,5.0,8.2,0.613,0.0,0.0,0.000,5.0,8.1,0.614,0.613,1.1,2.3,0.473,3.8,8.1,11.9,1.2,0.7,1.3,0.6,2.2,11.1
7,8,Wes Iwundu,27,3,1,27.3,2.7,6.0,0.444,1.0,1.7,0.600,1.7,4.3,0.385,0.528,1.0,1.3,0.750,1.0,3.3,4.3,0.0,0.3,0.0,0.3,2.7,7.3
8,9,Danilo Gallinari,33,66,18,25.3,3.9,9.0,0.434,1.7,4.5,0.381,2.2,4.6,0.485,0.528,2.1,2.4,0.904,0.5,4.1,4.7,1.5,0.4,0.2,0.6,1.4,11.7
9,10,Cam Reddish,22,34,7,23.4,4.1,10.1,0.402,1.7,4.5,0.379,2.4,5.6,0.421,0.487,2.1,2.4,0.900,0.6,1.9,2.5,1.1,1.0,0.3,1.3,1.3,11.9


In [ ]:
check_missing_values(dictionary_of_players)

In [ ]:
#celda de limpieza de datos
dictionary_of_players['ATL'][2022]['roster'] = dictionary_of_players['ATL'][2022]['roster'].drop(columns=['Unnamed: 6'])
dictionary_of_players['ATL'][2022]['team_and_opponent'] = dictionary_of_players['ATL'][2022]['team_and_opponent'].rename(columns={'Unnamed: 0': ''})
dictionary_of_players['ATL'][2022]['team_misc'].columns = dictionary_of_players['ATL'][2022]['team_misc'].iloc[0]
dictionary_of_players['ATL'][2022]['team_misc'] = dictionary_of_players['ATL'][2022]['team_misc'][1:]
dictionary_of_players['ATL'][2022]['per_poss'] = dictionary_of_players['ATL'][2022]['per_poss'] .drop(columns=['Unnamed: 27'])
dictionary_of_players['ATL'][2022]['advanced'] = dictionary_of_players['ATL'][2022]['advanced'].drop(columns=['Unnamed: 17', 'Unnamed: 22'])

In [ ]:
#pabajo players vs equipos

In [8]:
# url = 'https://www.basketball-reference.com/players/a/'
# driver.get(url)
# time.sleep(3)
# table_id = getTableIDS(url)[0]
# table_element = driver.find_element(By.ID, table_id)
# table_html = table_element.get_attribute('outerHTML')
# soup = BeautifulSoup(table_html, 'html.parser')
# filas = soup.find_all('tr')

# active_players = []
# for fila in filas:
#     if fila.find('strong'):
#         regex = re.compile(r'(?<=href=").*?(?=")')
#         href = regex.findall(str(fila))
#         href = href[0]
#         regex = re.compile(r'(?<=/).*(?=.html)')
#         player = regex.findall(href)
#         player = player[0]
#         resultado = re.search(r'[^/]+/([^/]+)$', player)
#         active_players.append(resultado.group(1))

# # Obtener la información de cada jugador activo
# dictionary_of_players_individually = {}

# for player in active_players:
#     url = f'https://www.basketball-reference.com/players/a/{player}.html'
#     driver.get(url)
#     time.sleep(1)  # Ajusta este tiempo según lo necesario, puede que no necesites tanto como 3 segundos
    
#     player_html = driver.page_source
#     player_soup = BeautifulSoup(player_html, 'html.parser')
    
#     player_name = player_soup.find("div", {"id": "info"}).find("span").text.strip()
#     # Aquí puedes obtener otros datos del jugador según tu necesidad y agregarlos al diccionario
#     dictionary_of_players_individually[player_name] = {}  # Agregar los datos del jugador

# # Ahora tienes un diccionario con la información de cada jugador activo
# print(dictionary_of_players_individually)


In [ ]:
# url = 'https://www.basketball-reference.com/players/a/'
# driver.get(url)
# time.sleep(3)
# table_id = getTableIDS(url)[0]
# table_element = driver.find_element(By.ID, table_id)
# table_html = table_element.get_attribute('outerHTML')
# soup = BeautifulSoup(table_html, 'html.parser')
# filas = soup.find_all('tr')
# dictionary_of_players_individually = {}

# active_players = []
# for fila in filas:
#     if fila.find('strong'):
#         regex = re.compile(r'(?<=href=").*?(?=")')
#         href = regex.findall(str(fila))
#         href = href[0]
#         regex = re.compile(r'(?<=/).*(?=.html)')
#         player = regex.findall(href)
#         player = player[0]
#         resultado = re.search(r'[^/]+/([^/]+)$', player)
#         active_players.append(resultado.group(1))
        
# for player in active_players:
#     url = f'https://www.basketball-reference.com/players/a/{player}.html'
#     driver.get(url)
#     time.sleep(3)
#     player_html = driver.page_source
#     player_soup = BeautifulSoup(player_html, 'html.parser')
#     player_name = player_soup.find("div", {"id": "info"}).find("span").text.strip()
#     table_ids = getTableIDS(url)

In [ ]:
'''Para mi yo del futuro
la celda de abajo es el diccionario que querias hacer de 
letra a: todos los jugadores con ese apellido y sus estadisticas
y luego la otra celda es de limpieza
lo de arriba entiendo yo que si se acaba haciendo lo de abajo se acabara borrando'''

In [9]:
# URL base de la página
base_url = 'https://www.basketball-reference.com/players/'

# Letra que quieres buscar
letra = 'a'

# URL de la página de jugadores con la letra específica
url = f'{base_url}{letra}/'

driver.get(url)
time.sleep(3)

table_id = getTableIDS(url)[0]
table_element = driver.find_element(By.ID, table_id)
table_html = table_element.get_attribute('outerHTML')
soup = BeautifulSoup(table_html, 'html.parser')
filas = soup.find_all('tr')

active_players = []

for fila in filas:
    if fila.find('strong'):
        regex = re.compile(r'(?<=href=").*?(?=")')
        href = regex.findall(str(fila))
        href = href[0]
        regex = re.compile(r'(?<=/).*(?=.html)')
        player = regex.findall(href)
        player = player[0]
        resultado = re.search(r'[^/]+/([^/]+)$', player)
        active_players.append(resultado.group(1))

# Diccionario para almacenar las estadísticas de los jugadores con apellido que empieza por 'A'
stats_by_letter = {}

# Obtener las estadísticas de cada jugador activo
for player in active_players:
    player_url = f'{base_url}{letra}/{player}.html'
    driver.get(player_url)
    time.sleep(3)
    player_html = driver.page_source
    player_soup = BeautifulSoup(player_html, 'html.parser')
    player_name = player_soup.find("div", {"id": "info"}).find("span").text.strip()
    table_ids = getTableIDS(player_url)
    player_stats = {}
    for table_id in table_ids:
        table_element = driver.find_element(By.ID, table_id)
        table_html = table_element.get_attribute('outerHTML')
        df = pd.read_html(table_html, header=0)[0]
        player_stats[table_id] = df

    stats_by_letter.setdefault(letra, {})[player_name] = player_stats

In [14]:
# Supongamos que tienes el diccionario stats_by_letter con las estadísticas de los jugadores

# Iterar sobre las claves principales del diccionario (en este caso, la letra 'A')
for letra, players_stats in stats_by_letter.items():
    print(f"Letra: {letra}")

    # Iterar sobre los jugadores y sus estadísticas
    for player, stats in players_stats.items():
        print(f"Jugador: {player}")
        # Imprimir las estadísticas de cada jugador
        for stat_name, value in stats.items():
            print(f"{stat_name}")

        print("\n")


In [12]:
year = 2024
team = 'ATL'
url = f'https://www.basketball-reference.com/teams/{team}/{year}.html'
table_ids = getTableIDS(url)

table_element = driver.find_element(By.ID, table_ids[0])
table_html = table_element.get_attribute('outerHTML')

In [13]:
table = table_ids[0]
if table:
    headers = [th.text.strip() for th in table.find('thead').find_all('th')]
    rows = [
            [cell.text.strip() for cell in tr.find_all(['th', 'td'])]
            for tr in table.find('tbody').find_all('tr', class_='full_table')
        ]

'<table class="sortable stats_table now_sortable" id="roster" data-cols-to-freeze=",2">\n    <caption>Roster Table</caption>\n    \n\n   <colgroup><col><col><col><col><col><col><col><col><col></colgroup>\n   <thead>      \n      <tr>\n         <th aria-label="No." data-stat="number" scope="col" class=" poptip sort_default_asc center" data-tip="Uniform Number">No.</th>\n         <th aria-label="Player" data-stat="player" scope="col" class=" poptip sort_default_asc center">Player</th>\n         <th aria-label="Pos" data-stat="pos" scope="col" class=" poptip sort_default_asc center" data-tip="Position">Pos</th>\n         <th aria-label="Ht" data-stat="height" scope="col" class=" poptip sort_default_asc center" data-tip="Height">Ht</th>\n         <th aria-label="Wt" data-stat="weight" scope="col" class=" poptip sort_default_asc center" data-tip="Weight">Wt</th>\n         <th aria-label="Birth Date" data-stat="birth_date" scope="col" class=" poptip sort_default_asc center">Birth Date</th>\n

In [ ]:
#tablas que borrar:
#last5games
#stathead_insights
for letra, players_stats in stats_by_letter.items():
    for player, stats in players_stats.items():
        stats_by_letter[letra][player][''] = delete_unnamed_columns(stats_by_letter[letra][player]['per_game'])

In [ ]:
#Quiero hacer un diccionario tal que sea como
#diccionario = {a: Giannis Antetokounmpo: stats, Steven Adams: stats, etc}

In [ ]:
#todo
# limpiar el script
# creo que no hace falta selenium? probar a sacar todos los datos sin selenium
# sacar datos totales de cada jugador por season (check)
# sacar datos de cada jugador por equipo (check?)
# sacar datos de cada equipo por season
# asi de momento, a lo mejor hacer algo con sqlite mas adelante (no sqlite)
# separar home results away results
